# Check for the way of combining predictions in ensemble_6 (5 models), ensemble_4 (6 models), ensemble_5 (7 models) 

In [2]:
import os
import pandas as pd
import numpy as np
from adbench.myutils import Utils
import sklearn.metrics as skm
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
ensemble_6_dir = "../results/adbench/ensemble_6/train_seed=0"
ensemble_4_dir = "../results/adbench/ensemble_4/train_seed=0"
ensemble_5_dir = "../results/adbench/ensemble_5/train_seed=0"

In [4]:
def low_density_anomalies(test_log_probs, num_anomalies):
    """ Helper function for the F1-score, selects the num_anomalies lowest values of test_log_prob
    """
    anomaly_indices = np.argpartition(test_log_probs, num_anomalies - 1)[:num_anomalies]
    preds = np.zeros(len(test_log_probs))
    preds[anomaly_indices] = 1
    return preds

In [5]:
seed = 0
utils = Utils()  # utils function
utils.set_seed(seed)

### 1. For the situation that we compute the ensemble `average score per sample`

In [9]:
ensemble_aucroc_scores = []
ensemble_aucpr_scores = []
ensemble_f1_scores = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(ensemble_dir):
    if "all_scores_seed0.npz" in files:
        file_path = os.path.join(subdir, "all_scores_seed0.npz")
        df = np.load(file_path)
        all_scores = df['all_scores']
        labels = df['labels']

        # compute the ensemble average score per sample
        avg_score = np.mean(all_scores, axis=0)

        # following DTE:
        indices = np.arange(len(labels))
        p = low_density_anomalies(-avg_score, len(indices[labels == 1]))
        f1_score = skm.f1_score(labels, p)
        inds = np.where(np.isnan(avg_score))
        avg_score[inds] = 0
        result = utils.metric(y_true=labels, y_score=avg_score)
        result = {seed: {'aucroc': result['aucroc'], 'aucpr': result['aucpr'], 'f1': f1_score,
                         'time-per-inst': 0,
                         'context-len': 0}}
        df = pd.DataFrame(result)

        # Transpose the DataFrame to match the desired structure
        df = df.transpose().reset_index()
        df.columns = ['seed', 'aucroc', 'aucpr', 'f1', 'time-per-inst', 'context-len']


        # extract scores
        aucroc = df['aucroc'].values[0]
        ensemble_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        ensemble_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        ensemble_f1_scores.append(f1)

# compute average scores of all datasets
ensemble_avg_aucroc = round(sum(ensemble_aucroc_scores) / len(ensemble_aucroc_scores), 2)
ensemble_avg_aucpr = round(sum(ensemble_aucpr_scores) / len(ensemble_aucpr_scores), 2)
ensemble_avg_f1 = round(sum(ensemble_f1_scores) / len(ensemble_f1_scores), 2)

print(f"number of datasets: {len(ensemble_aucroc_scores)}")
print(f"Average aucroc: {ensemble_avg_aucroc}")
print(f"Average aucpr: {ensemble_avg_aucpr}")
print(f"Average f1: {ensemble_avg_f1}")

number of datasets: 121
Average aucroc: 0.77
Average aucpr: 0.48
Average f1: 0.47


### 2. For the situation that we compute the ensemble `max score per sample`

In [10]:
max_ensemble_aucroc_scores = []
max_ensemble_aucpr_scores = []
max_ensemble_f1_scores = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(ensemble_dir):
    if "all_scores_seed0.npz" in files:
        file_path = os.path.join(subdir, "all_scores_seed0.npz")
        df = np.load(file_path)
        all_scores = df['all_scores']
        labels = df['labels']

        # compute the ensemble max score per sample
        max_score = np.max(all_scores, axis=0)

        # following DTE:
        indices = np.arange(len(labels))
        p = low_density_anomalies(-max_score, len(indices[labels == 1]))
        f1_score = skm.f1_score(labels, p)
        inds = np.where(np.isnan(max_score))
        max_score[inds] = 0
        result = utils.metric(y_true=labels, y_score=max_score)
        result = {seed: {'aucroc': result['aucroc'], 'aucpr': result['aucpr'], 'f1': f1_score,
                         'time-per-inst': 0,
                         'context-len': 0}}
        df = pd.DataFrame(result)

        # Transpose the DataFrame to match the desired structure
        df = df.transpose().reset_index()
        df.columns = ['seed', 'aucroc', 'aucpr', 'f1', 'time-per-inst', 'context-len']


        # extract scores
        aucroc = df['aucroc'].values[0]
        max_ensemble_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        max_ensemble_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        max_ensemble_f1_scores.append(f1)

# compute average scores of all datasets
max_ensemble_avg_aucroc = round(sum(max_ensemble_aucroc_scores) / len(max_ensemble_aucroc_scores), 2)
max_ensemble_avg_aucpr = round(sum(max_ensemble_aucpr_scores) / len(max_ensemble_aucpr_scores), 2)
max_ensemble_avg_f1 = round(sum(max_ensemble_f1_scores) / len(max_ensemble_f1_scores), 2)

print(f"number of datasets: {len(max_ensemble_aucroc_scores)}")
print(f"Average aucroc: {max_ensemble_avg_aucroc}")
print(f"Average aucpr: {max_ensemble_avg_aucpr}")
print(f"Average f1: {max_ensemble_avg_f1}")

number of datasets: 121
Average aucroc: 0.75
Average aucpr: 0.47
Average f1: 0.46


### 3. For the situation that we compute the ensemble `min score per sample`

In [11]:
min_ensemble_aucroc_scores = []
min_ensemble_aucpr_scores = []
min_ensemble_f1_scores = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(ensemble_dir):
    if "all_scores_seed0.npz" in files:
        file_path = os.path.join(subdir, "all_scores_seed0.npz")
        df = np.load(file_path)
        all_scores = df['all_scores']
        labels = df['labels']

        # compute the ensemble min score per sample
        min_score = np.min(all_scores, axis=0)

        # following DTE:
        indices = np.arange(len(labels))
        p = low_density_anomalies(-min_score, len(indices[labels == 1]))
        f1_score = skm.f1_score(labels, p)
        inds = np.where(np.isnan(min_score))
        min_score[inds] = 0
        result = utils.metric(y_true=labels, y_score=min_score)
        result = {seed: {'aucroc': result['aucroc'], 'aucpr': result['aucpr'], 'f1': f1_score,
                         'time-per-inst': 0,
                         'context-len': 0}}
        df = pd.DataFrame(result)

        # Transpose the DataFrame to match the desired structure
        df = df.transpose().reset_index()
        df.columns = ['seed', 'aucroc', 'aucpr', 'f1', 'time-per-inst', 'context-len']


        # extract scores
        aucroc = df['aucroc'].values[0]
        min_ensemble_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        min_ensemble_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        min_ensemble_f1_scores.append(f1)

# compute average scores of all datasets
min_ensemble_avg_aucroc = round(sum(min_ensemble_aucroc_scores) / len(min_ensemble_aucroc_scores), 2)
min_ensemble_avg_aucpr = round(sum(min_ensemble_aucpr_scores) / len(min_ensemble_aucpr_scores), 2)
min_ensemble_avg_f1 = round(sum(min_ensemble_f1_scores) / len(min_ensemble_f1_scores), 2)

print(f"number of datasets: {len(min_ensemble_aucroc_scores)}")
print(f"Average aucroc: {min_ensemble_avg_aucroc}")
print(f"Average aucpr: {min_ensemble_avg_aucpr}")
print(f"Average f1: {min_ensemble_avg_f1}")

number of datasets: 121
Average aucroc: 0.76
Average aucpr: 0.47
Average f1: 0.45


### 4. For the situation that we compute the ensemble `meam square score per sample`

In [7]:
ensemble_aucroc_scores = []
ensemble_aucpr_scores = []
ensemble_f1_scores = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(ensemble_dir):
    if "all_scores_seed0.npz" in files:
        file_path = os.path.join(subdir, "all_scores_seed0.npz")
        df = np.load(file_path)
        all_scores = df['all_scores']
        labels = df['labels']

        # compute the ensemble mean square score per sample
        avg_score = np.mean(all_scores**2, axis=0)

        # following DTE:
        indices = np.arange(len(labels))
        p = low_density_anomalies(-avg_score, len(indices[labels == 1]))
        f1_score = skm.f1_score(labels, p)
        inds = np.where(np.isnan(avg_score))
        avg_score[inds] = 0
        result = utils.metric(y_true=labels, y_score=avg_score)
        result = {seed: {'aucroc': result['aucroc'], 'aucpr': result['aucpr'], 'f1': f1_score,
                         'time-per-inst': 0,
                         'context-len': 0}}
        df = pd.DataFrame(result)

        # Transpose the DataFrame to match the desired structure
        df = df.transpose().reset_index()
        df.columns = ['seed', 'aucroc', 'aucpr', 'f1', 'time-per-inst', 'context-len']


        # extract scores
        aucroc = df['aucroc'].values[0]
        ensemble_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        ensemble_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        ensemble_f1_scores.append(f1)

# compute average scores of all datasets
ensemble_avg_aucroc = round(sum(ensemble_aucroc_scores) / len(ensemble_aucroc_scores), 2)
ensemble_avg_aucpr = round(sum(ensemble_aucpr_scores) / len(ensemble_aucpr_scores), 2)
ensemble_avg_f1 = round(sum(ensemble_f1_scores) / len(ensemble_f1_scores), 2)

print(f"number of datasets: {len(ensemble_aucroc_scores)}")
print(f"Average aucroc: {ensemble_avg_aucroc}")
print(f"Average aucpr: {ensemble_avg_aucpr}")
print(f"Average f1: {ensemble_avg_f1}")

number of datasets: 121
Average aucroc: 0.76
Average aucpr: 0.48
Average f1: 0.47


In [9]:
from sklearn.linear_model import LogisticRegression


In [13]:
ensemble_aucroc_scores = []
ensemble_aucpr_scores = []
ensemble_f1_scores = []
# walk through all folders and find result.csv files
for subdir, _, files in os.walk(ensemble_dir):
    if "all_scores_seed0.npz" in files:
        file_path = os.path.join(subdir, "all_scores_seed0.npz")
        df = np.load(file_path)
        all_scores = df['all_scores']
        labels = df['labels']

        # compute the ensemble mean square score per sample
        logistic_regression = LogisticRegression()
        logistic_regression.fit(all_scores.T, labels)
        avg_score = logistic_regression.predict_proba(all_scores.T)[:,1]
        print(logistic_regression.coef_)

        # following DTE:
        indices = np.arange(len(labels))
        p = low_density_anomalies(-avg_score, len(indices[labels == 1]))
        f1_score = skm.f1_score(labels, p)
        inds = np.where(np.isnan(avg_score))
        avg_score[inds] = 0
        result = utils.metric(y_true=labels, y_score=avg_score)
        result = {seed: {'aucroc': result['aucroc'], 'aucpr': result['aucpr'], 'f1': f1_score,
                         'time-per-inst': 0,
                         'context-len': 0}}
        df = pd.DataFrame(result)

        # Transpose the DataFrame to match the desired structure
        df = df.transpose().reset_index()
        df.columns = ['seed', 'aucroc', 'aucpr', 'f1', 'time-per-inst', 'context-len']


        # extract scores
        aucroc = df['aucroc'].values[0]
        ensemble_aucroc_scores.append(aucroc)

        aucpr = df['aucpr'].values[0]
        ensemble_aucpr_scores.append(aucpr)

        f1 = df['f1'].values[0]
        ensemble_f1_scores.append(f1)

# compute average scores of all datasets
ensemble_avg_aucroc = round(sum(ensemble_aucroc_scores) / len(ensemble_aucroc_scores), 2)
ensemble_avg_aucpr = round(sum(ensemble_aucpr_scores) / len(ensemble_aucpr_scores), 2)
ensemble_avg_f1 = round(sum(ensemble_f1_scores) / len(ensemble_f1_scores), 2)

print(f"number of datasets: {len(ensemble_aucroc_scores)}")
print(f"Average aucroc: {ensemble_avg_aucroc}")
print(f"Average aucpr: {ensemble_avg_aucpr}")
print(f"Average f1: {ensemble_avg_f1}")

[[-1.08378593 -0.13682696  6.42602418 -1.68103924  1.19092502 -0.59127828]]
[[-0.13838667  0.71542334  0.51207339  2.48812385  0.12306335  0.4405618 ]]
[[-0.61670163  0.2020102   1.73165315  0.69914866 -0.43784818  2.0540996 ]]
[[1.44552586 0.61873323 0.49079531 1.70185635 3.23867637 0.30154375]]
[[ 6.74925808 -1.54989545  2.91660553 -1.37314481 -0.27339365 -2.30163887]]
[[ 3.68723871  1.5116507   4.03454373  2.06485478 -0.78646528  3.94310407]]
[[ 1.68908285  1.85042778  1.52447082  0.55032306 -2.41726847  2.34146538]]
[[2.39075947 0.41382985 0.73073039 0.35550546 3.34673804 0.37651316]]
[[ 1.08365176  0.10446785  4.26306269  1.72958544  2.89088534 -1.17687849]]
[[ 0.19754166  0.75562701 -1.2800031  -0.22316244 -0.44836515  2.29836752]]
[[ 0.52029033  0.85753941  1.54907825 -2.18614723 -0.15494959  0.07870958]]
[[0.07047531 0.94903416 1.02778792 2.3506324  0.16913731 0.15851467]]
[[ 1.98168634  0.35107584  0.10187584 -0.03997365 -0.18284287 -0.26366203]]
[[ 0.16019201  8.1401074   1.0

In [10]:
print(all_scores.shape)

(6, 2734)
